# 📋 Day 03a: Template Method & Iterator Patterns

---

## 🎯 What You'll Master Today
- **Template Method**: define algorithm skeleton, let subclasses fill in steps
- **Iterator**: traverse a collection without exposing its internals
- Python's built-in iterator protocol (`__iter__`, `__next__`, generators)

---

## 🎭 The Analogies

**Template Method** = Recipe template  
"Boil water → Add ingredient → Cook X minutes → Serve." The steps are fixed. But WHAT ingredient and HOW you serve varies — tea vs coffee.

**Iterator** = TV remote channel surfing  
You press "Next" to go through channels. You don't care if channels are stored in an array, linked list, or satellite feed. You just press Next.

---

In [ ]:
# ============================================
# 1. TEMPLATE METHOD — Data Processing Pipeline
# ============================================
from abc import ABC, abstractmethod

class DataProcessor(ABC):
    """Template — fixed algorithm, customizable steps."""
    
    def process(self, source: str) -> dict:
        """The template method — DON'T override this."""
        raw = self.extract(source)
        clean = self.transform(raw)
        result = self.load(clean)
        self.notify(result)  # hook — optional override
        return result
    
    @abstractmethod
    def extract(self, source: str) -> str: pass
    
    @abstractmethod
    def transform(self, data: str) -> dict: pass
    
    @abstractmethod
    def load(self, data: dict) -> dict: pass
    
    def notify(self, result: dict):
        """Hook method — optional override."""
        pass


class CSVProcessor(DataProcessor):
    def extract(self, source):
        print(f"  📂 Reading CSV from {source}")
        return "name,age\nAlice,30\nBob,25"
    
    def transform(self, data):
        lines = data.strip().split("\n")
        headers = lines[0].split(",")
        rows = [dict(zip(headers, line.split(","))) for line in lines[1:]]
        print(f"  🔄 Parsed {len(rows)} rows")
        return {"rows": rows, "count": len(rows)}
    
    def load(self, data):
        print(f"  💾 Saved {data['count']} records to database")
        return {"status": "success", **data}


class APIProcessor(DataProcessor):
    def extract(self, source):
        print(f"  🌐 Fetching from API: {source}")
        return '{"users": [{"name": "Charlie"}]}'
    
    def transform(self, data):
        import json
        parsed = json.loads(data)
        print(f"  🔄 Transformed {len(parsed['users'])} users")
        return parsed
    
    def load(self, data):
        print(f"  💾 Cached API response")
        return {"status": "cached", **data}
    
    def notify(self, result):  # override hook
        print(f"  📧 Notified team about API sync")


print("CSV Pipeline:")
CSVProcessor().process("data.csv")

print("\nAPI Pipeline:")
APIProcessor().process("https://api.example.com/users")

In [ ]:
# ============================================
# 2. ITERATOR — Custom Collection
# ============================================

class NumberRange:
    """Custom iterable — implements iterator protocol."""
    
    def __init__(self, start: int, end: int, step: int = 1):
        self.start = start
        self.end = end
        self.step = step
    
    def __iter__(self):
        """Returns an iterator object."""
        self._current = self.start
        return self
    
    def __next__(self):
        """Returns next value or raises StopIteration."""
        if self._current >= self.end:
            raise StopIteration
        value = self._current
        self._current += self.step
        return value


# Works with for loop, list(), sum(), etc.
print("For loop:")
for n in NumberRange(0, 10, 3):
    print(f"  {n}", end="")
print()

print(f"\nAs list: {list(NumberRange(1, 6))}")
print(f"Sum:     {sum(NumberRange(1, 11))}")

In [ ]:
# ============================================
# 3. Generators — Pythonic Iterators
# ============================================

# Generator function — yield makes it an iterator
def fibonacci(limit: int):
    a, b = 0, 1
    while a < limit:
        yield a
        a, b = b, a + b

print("Fibonacci < 100:")
print(list(fibonacci(100)))

# Generator expression — like list comprehension but lazy
squares = (x**2 for x in range(10))  # no memory allocated!
print(f"\nSquares: {list(squares)}")

# Infinite generator — memory efficient
def infinite_ids(prefix: str = "ID"):
    counter = 1
    while True:
        yield f"{prefix}-{counter:04d}"
        counter += 1

gen = infinite_ids("USER")
print(f"\nGenerated IDs: {[next(gen) for _ in range(5)]}")

In [ ]:
# ============================================
# 4. Separate Iterator from Collection
# ============================================

class Playlist:
    """Collection — returns different iterators."""
    def __init__(self, songs: list[str]):
        self._songs = songs
    
    def __iter__(self):
        """Default: forward iteration."""
        return iter(self._songs)
    
    def reverse(self):
        """Reverse iterator."""
        return reversed(self._songs)
    
    def shuffle(self):
        """Shuffled iterator."""
        import random
        shuffled = self._songs[:]
        random.shuffle(shuffled)
        return iter(shuffled)


playlist = Playlist(["Song A", "Song B", "Song C", "Song D"])

print("Normal:  ", list(playlist))
print("Reversed:", list(playlist.reverse()))
print("Shuffled:", list(playlist.shuffle()))

```
╔═════════════════════════════════════════════════════════╗
║         PYTHON ITERATOR PROTOCOL CHEAT SHEET           ║
╠═════════════════════════════════════════════════════════╣
║                                                         ║
║  Iterable: has __iter__() → returns iterator            ║
║  Iterator: has __next__() → returns next value           ║
║  Generator: function with yield (auto-iterator)          ║
║                                                         ║
║  for x in obj:   →   iter(obj) → __next__() loop       ║
║  list(obj)       →   same, collects all values          ║
║  next(it)        →   one __next__() call                ║
║                                                         ║
╚═════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | Template Method vs Strategy? | Template: fixed skeleton with overridable steps (inheritance). Strategy: swap entire algorithm (composition) |
| 2 | What's a hook method? | Optional step in template — default implementation (often empty), subclass may override |
| 3 | Iterable vs Iterator? | Iterable has `__iter__`. Iterator has `__next__`. All iterators are iterable, not vice versa |
| 4 | Generator vs list? | Generator is lazy — O(1) memory. List stores all values — O(n) memory |
| 5 | When to use Template Method? | When algorithm structure is fixed but steps vary. ETL pipelines, test frameworks |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Template Method = fixed skeleton + variable steps     │
## │  • Hook methods = optional overrides with defaults       │
## │  • Iterator = traverse without exposing internals        │
## │  • Python: __iter__ + __next__ = iterator protocol       │
## │  • Generators are the Pythonic way to create iterators   │
## │  • yield = lazy, memory-efficient iteration              │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Chain of Responsibility & Mediator** — passing requests along a chain and centralizing communication